In [0]:
dbutils.widgets.text("catalog", "claims_dev")

In [0]:
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold.dim_date AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d AS date,
  year(d) AS year,
  quarter(d) AS quarter,
  month(d) AS month,
  date_format(d, 'MMMM') AS month_name,
  dayofweek(d) AS day_of_week,
  dayofweek(d) IN (1, 7) AS is_weekend
FROM (SELECT explode(sequence(DATE'2025-01-01', DATE'2026-12-31')) AS d)
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold.dim_policy AS
SELECT
  xxhash64(policy_id, __START_AT) AS policy_sk,
  policy_id, customer_id, plan, vehicle_year, vehicle_value, annual_premium,
  state, city, start_date,
  __START_AT AS valid_from,
  coalesce(__END_AT, TIMESTAMP'9999-12-31') AS valid_to,
  __END_AT IS NULL AS is_current
FROM silver.policies
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold.dim_workshop AS
SELECT workshop_id, workshop_name, state, city, network, CAST(opened_date AS DATE) AS opened_date
FROM bronze.workshops
""")

spark.sql("""
CREATE OR REPLACE TABLE gold.dim_coverage AS
SELECT DISTINCT coverage_code, coverage_name, CAST(deductible_pct AS DECIMAL(4,2)) AS deductible_pct
FROM bronze.coverages
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold.fact_claim AS
WITH c AS (
  SELECT
    claim_id,
    any_value(policy_id) AS policy_id,
    any_value(workshop_id) AS workshop_id,
    any_value(coverage_code) AS coverage_code,
    any_value(cause) AS cause,
    any_value(state) AS state,
    any_value(city) AS city,
    any_value(source) AS source,
    max(channel) AS channel,
    min(event_ts) AS reported_ts,
    min(event_ts) FILTER (WHERE event_type = 'ADJUSTMENT') AS adjusted_ts,
    min(event_ts) FILTER (WHERE event_type = 'PAYMENT') AS paid_ts,
    min(event_ts) FILTER (WHERE event_type = 'CLOSE') AS closed_ts,
    max(amount) FILTER (WHERE event_type = 'ADJUSTMENT') AS estimate_amount,
    sum(amount) FILTER (WHERE event_type = 'PAYMENT') AS paid_amount,
    coalesce(max(status) FILTER (WHERE event_type = 'CLOSE'), 'open') AS status
  FROM silver.claim_events
  GROUP BY claim_id
)
SELECT
  c.claim_id,
  p.policy_sk,
  c.policy_id,
  c.workshop_id,
  c.coverage_code,
  CAST(date_format(c.reported_ts, 'yyyyMMdd') AS INT) AS reported_date_key,
  c.cause, c.state, c.city, c.source, c.channel,
  c.reported_ts, c.adjusted_ts, c.paid_ts, c.closed_ts,
  c.estimate_amount, c.paid_amount, c.status,
  datediff(c.closed_ts, c.reported_ts) AS days_to_close,
  p.policy_sk IS NOT NULL AS policy_in_force
FROM c
LEFT JOIN gold.dim_policy p
  ON c.policy_id = p.policy_id
 AND c.reported_ts >= p.valid_from
 AND c.reported_ts < p.valid_to
""")

In [0]:
display(spark.sql("""
SELECT 'dim_date' AS t, count(*) AS n FROM gold.dim_date
UNION ALL SELECT 'dim_policy', count(*) FROM gold.dim_policy
UNION ALL SELECT 'dim_policy_current', count_if(is_current) FROM gold.dim_policy
UNION ALL SELECT 'dim_workshop', count(*) FROM gold.dim_workshop
UNION ALL SELECT 'dim_coverage', count(*) FROM gold.dim_coverage
UNION ALL SELECT 'fact_claim', count(*) FROM gold.fact_claim
UNION ALL SELECT 'without_policy', count_if(NOT policy_in_force) FROM gold.fact_claim
UNION ALL SELECT 'without_workshop', count_if(workshop_id IS NULL) FROM gold.fact_claim
"""))
display(spark.sql("""
SELECT status, count(*) AS claims, sum(estimate_amount) AS estimate, sum(paid_amount) AS paid
FROM gold.fact_claim
GROUP BY status
ORDER BY claims DESC
"""))